In [1]:
%pip install qutip

Note: you may need to restart the kernel to use updated packages.


#	Build $H_{\rm eff}^b$

$H_{\rm eff}^b=T+U$, where $$T = J_x \sum_i \big(\hat a_i^\dagger \hat a_{i+1} + \text{h.c.}\big), \quad U = \sum_i V(i)\hat n_i + \sum_{i<j} W(i,j)\hat n_i \hat n_j.$$

In [40]:
import numpy as np
import qutip as qt

def site_op(op, i, N):
    ops = [qt.qeye(2)] * N
    ops[i] = op
    return qt.tensor(ops)

def build_Heff(N, alpha, Jz, Jx):
    # operators on every site, built once and reused
    a = [site_op(qt.destroy(2), i, N) for i in range(N)]
    n = [site_op(qt.num(2), i, N) for i in range(N)]

    # coupling 1/|i-j|^alpha
    def c(i, j):
        return 1.0 / abs(i - j)**alpha

    # confining potential V(i) = -2 Jz * sum_{j != i} 1/|i-j|^alpha
    V = [-2 * Jz * sum(c(i, j) for j in range(N) if j != i) for i in range(N)]

    H = 0
    # T: nearest-neighbour hopping, open chain (bonds 0-1, 1-2, ..., (N-2)-(N-1))
    for i in range(N - 1):
        H += Jx * (a[i].dag() * a[i+1] + a[i+1].dag() * a[i])
    # U, part 1: potential
    for i in range(N):
        H += V[i] * n[i]
    # U, part 2: repulsion W(i,j) = 4 Jz / |i-j|^alpha
    for i in range(N):
        for j in range(i + 1, N):
            H += 4 * Jz * c(i, j) * n[i] * n[j]
    return H

N, alpha, Jz, Jx, b = 6, 0.5, 1.0, 0.1, 2
H = build_Heff(N, alpha, Jz, Jx)

idx = [k for k in range(2**N) if bin(k).count('1') == b]   # indices with b ones
Hb = qt.Qobj(H.full()[np.ix_(idx, idx)])                    # keep those rows and columns
Ntot = sum(site_op(qt.num(2), i, N) for i in range(N))
print("Hermitian:     ", H.isherm)
print("[H, Ntot]:     ", (H * Ntot - Ntot * H).norm())
print("[H, R]:        ", (H.permute(list(range(N))[::-1]) - H).norm())
print("block shape:   ", Hb.shape)
print("lowest 3 E:    ", np.round(Hb.eigenenergies()[:3], 4))
print("first state:   ", format(idx[0], '06b'), round(Hb.diag()[0], 4))

Hermitian:      True
[H, Ntot]:      0.0
[H, R]:         0.0
block shape:    (15, 15)
lowest 3 E:     [-12.9569 -12.7493 -12.6699]
first state:    000011 -10.0323


For the optimization we won't build big matrices. We only need the diagonal energy of each configuration:

$$E(c)=\sum_{i\in c}V(i)+\sum_{i<j\in c}W(i,j).$$

We store $V$ as an array and $W$ as an $N\times N$ matrix, then add up the right entries.

In [41]:
def get_V_W(N, alpha, Jz):
    V = np.zeros(N)          # V[i]
    W = np.zeros((N, N))     # W[i, j]
    for i in range(N):
        for j in range(N):
            if i != j:
                V[i] += -2 * Jz / abs(i - j)**alpha
                W[i, j] = 4 * Jz / abs(i - j)**alpha
    return V, W

def config_energy(occ, V, W):
    """occ = list of occupied sites, e.g. [4, 5]"""
    E = sum(V[i] for i in occ)                  # potential of each excitation
    for x in range(len(occ)):
        for y in range(x + 1, len(occ)):        # every pair of excitations, counted once
            E += W[occ[x], occ[y]]
    return E

Matching the QuTiP diagonal to 0.0 proves the shortcut is right. Anything we build on it later inherits that guarantee.

In [43]:
V, W = get_V_W(N, alpha, Jz)
print(np.round(V, 4))
print(W[4, 5])
print(config_energy([4, 5], V, W))

def occupied(k, N):
    bits = format(k, f'0{N}b')                  # e.g. 3 -> '000011'
    return [i for i, bit in enumerate(bits) if bit == '1']

E_fast = [config_energy(occupied(k, N), V, W) for k in idx]
print(np.abs(np.array(E_fast) - Hb.diag()).max())

[-6.4633 -7.5689 -7.9831 -7.9831 -7.5689 -6.4633]
4.0
-10.03225539250461
0.0


In [44]:
import itertools
from math import comb

def config_from_S(S, t, N):
    occ = []
    for p, tp in enumerate(t):
        L, R = p, N - 1 - p          # left and right site of pair p
        if tp == 2:
            occ += [L, R]            # doubly occupied: both sites
        elif tp == 1:
            occ.append(L if p in S else R)   # active: left if p is in S, else right
        # tp == 0: empty, add nothing
    return sorted(occ)

def codewords(t):
    A = [p for p, tp in enumerate(t) if tp == 1]
    K = len(A) // 2
    words = [set(S) for S in itertools.combinations(A, K) if A[0] in S]
    return words, A, K

N, alpha, Jz = 12, 0.5, 1.0
t = [1, 1, 1, 1, 0, 0]          # pairs 0-3 active, pairs 4-5 empty
words, A, K = codewords(t)
print("A =", A, " K =", K, " dim =", len(words), " expected =", comb(2*K, K)//2)

V, W = get_V_W(N, alpha, Jz)
for S in words:
    Sc   = set(A) - S
    occS = config_from_S(S, t, N)
    bits = ''.join('1' if i in occS else '0' for i in range(N))
    M    = sum(i >= N//2 for i in occS) - sum(i < N//2 for i in occS)
    print(sorted(S), bits, "M =", M,
          " E(S) =", round(config_energy(occS, V, W), 4),
          " E(Sc) =", round(config_energy(config_from_S(Sc, t, N), V, W), 4))

A = [0, 1, 2, 3]  K = 2  dim = 3  expected = 3
[0, 1] 110000001100 M = 0  E(S) = -35.1489  E(Sc) = -35.1489
[0, 2] 101000001010 M = 0  E(S) = -37.4394  E(Sc) = -37.4394
[0, 3] 100100000110 M = 0  E(S) = -36.7701  E(Sc) = -36.7701


Listing the codewords. itertools.combinations(A, K) gives every way to pick $K$ pairs out of A. But $S$ and its complement $S^c$ are the same codeword, since $|v_S\rangle=|v_{S^c}\rangle$. So we keep only one of each by requiring that the first active pair is in $S$: